# FAM vs corner-WFS correlations at the 4 corners (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-07-07  
**Last Modified:** 2026-07-07  
**Status:** In Progress  
**Keywords:** AOS, corner WFS, FAM, Zernike, correlation  

## Description

Sample notebook to explore the per-corner FAM-vs-CWFS comparison written by
the `wfs_corner_compare` pipeline step (`run_wfs_corner_compare.py`).  For each
FAM triplet, each corner raft (R00/R04/R40/R44) and each Noll Zernike j, the
step records the FAM measured OPD interpolated to the corner azimuth
(`fam_interp`) and the CWFS median at that corner (`cwfs_median`), plus sidecar
columns (rotator angle, elevation, mjd, science program).

Key functionality:
1. Load one CWFS variant's `wfs_corner_compare.parquet` (tidy: 1 row per triplet×corner×j).
2. FAM-vs-CWFS scatter per corner for a chosen set of Zernikes, with Pearson r +
   OLS line, optionally coloured by a sidecar variable (rotator/elevation/day).
3. Time history (vs day_obs) of FAM and CWFS for a chosen (Zj, corner).
4. Per-Zj summary of correlation r and OLS slope across the 4 corners.

**Output:** inline plots only (no files).

**Based on:** `code/run_wfs_corner_compare.py` (the `wfs_corner_compare` step).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-07 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Load](#data)
4. [Correlation scatter per corner](#scatter)
5. [Time history](#timehist)
6. [Per-Zj summary](#summary)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================
PARAM_SET = 'fam_danish_1_2_0_wep17_6_1_refitWCS_bin2x'
WFS_NAME  = 'refitWcs'          # CWFS variant: refitWcs | paired_3mm | ai_donut
OUTPUT_ROOT = 'output'          # aos/output (symlink on RSP)

# Which Zernikes / corners to plot in the scatter grid
ZJ_LIST = [5, 6, 7, 8, 11]      # Noll indices (subset of 4..19,22..26)
CORNERS = ['R00_SW0', 'R04_SW0', 'R40_SW0', 'R44_SW0']

# Colour the scatter by a sidecar column (None = single colour)
COLOR_BY = 'rotator_angle'      # 'rotator_angle' | 'alt_deg' | 'day_obs' | 'mjd' | None

# Optional selection cuts (None = no cut)
PROGRAM_FILTER = None           # e.g. 'BLOCK-T614_triplets' to keep only in-family triplets
ROT_ABS_MAX = None              # e.g. 65.0 to drop |rotator| beyond this

# Time-history pick
TH_ZJ = 5
TH_CORNER = 'R00_SW0'

<a id='setup'></a>
## Setup & Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from lsst.ts.intrinsic.wavefront.common.zernike_names import NOLL_NAMES
except Exception:
    NOLL_NAMES = {}


def robust_line(x, y):
    """Finite-only Pearson r + OLS slope/intercept.  Returns dict."""
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return dict(n=int(m.sum()), r=np.nan, slope=np.nan, off=np.nan)
    xk, yk = np.asarray(x)[m], np.asarray(y)[m]
    s, b = np.polyfit(xk, yk, 1)
    return dict(n=int(m.sum()), r=float(np.corrcoef(xk, yk)[0, 1]),
                slope=float(s), off=float(b))

<a id='data'></a>
## Load

In [ ]:
pq_path = Path(OUTPUT_ROOT) / PARAM_SET / 'wfs' / WFS_NAME / 'wfs_corner_compare.parquet'
df = pd.read_parquet(pq_path)
print(f'loaded {pq_path}')
print(f'  {len(df):,} rows; {df.groupby(["day_obs", "seq_num"]).ngroups} triplets; '
      f'corners={sorted(df.corner.unique())}; j={sorted(df.j.unique())}')
print(f'  columns: {list(df.columns)}')

# optional cuts
if PROGRAM_FILTER is not None:
    df = df[df.science_program == PROGRAM_FILTER]
if ROT_ABS_MAX is not None:
    df = df[df.rotator_angle.abs() <= ROT_ABS_MAX]
print(f'  after cuts: {len(df):,} rows')
df.head()

<a id='scatter'></a>
## Correlation scatter per corner

Rows = corners, columns = Zernikes.  x = FAM interpolated to the corner, y = CWFS
median.  Dashed line = 1:1, solid = OLS fit; title gives Pearson r and slope.

In [ ]:
nr, nc = len(CORNERS), len(ZJ_LIST)
fig, axs = plt.subplots(nr, nc, figsize=(3.0 * nc, 3.0 * nr), squeeze=False,
                        constrained_layout=True)
for ri, corner in enumerate(CORNERS):
    for ci, j in enumerate(ZJ_LIST):
        ax = axs[ri][ci]
        sub = df[(df.corner == corner) & (df.j == j)]
        x, y = sub.fam_interp.to_numpy(), sub.cwfs_median.to_numpy()
        fit = robust_line(x, y)
        c = sub[COLOR_BY] if (COLOR_BY and COLOR_BY in sub) else None
        sc = ax.scatter(x, y, s=8, alpha=0.5, c=c, cmap='turbo')
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() >= 3:
            lo, hi = np.nanpercentile(np.concatenate([x[m], y[m]]), [1, 99])
            ax.plot([lo, hi], [lo, hi], 'k--', lw=0.7)
            xs = np.array([lo, hi])
            ax.plot(xs, fit['slope'] * xs + fit['off'], 'r-', lw=0.9)
        nm = NOLL_NAMES.get(j, f'Z{j}')
        ax.set_title(f'{corner} Z{j} {nm}\nr={fit["r"]:.2f} m={fit["slope"]:.2f} n={fit["n"]}',
                     fontsize=7)
        ax.tick_params(labelsize=6)
        if ci == 0:
            ax.set_ylabel('CWFS median [µm]', fontsize=7)
        if ri == nr - 1:
            ax.set_xlabel('FAM interp [µm]', fontsize=7)
if COLOR_BY:
    fig.colorbar(sc, ax=axs, shrink=0.5, label=COLOR_BY)
fig.suptitle(f'FAM vs CWFS at the corners — {WFS_NAME} ({PARAM_SET})', fontsize=11)
plt.show()

<a id='timehist'></a>
## Time history

In [ ]:
sub = df[(df.corner == TH_CORNER) & (df.j == TH_ZJ)].sort_values(['day_obs', 'seq_num'])
x = np.arange(len(sub))
fig, ax = plt.subplots(figsize=(13, 4), constrained_layout=True)
ax.plot(x, sub.fam_interp, '.', ms=4, label='FAM interp')
ax.plot(x, sub.cwfs_median, '.', ms=4, label='CWFS median')
# day boundaries
days = sub.day_obs.to_numpy()
for b in np.where(np.diff(days) != 0)[0]:
    ax.axvline(b + 0.5, color='0.8', lw=0.6)
nm = NOLL_NAMES.get(TH_ZJ, f'Z{TH_ZJ}')
ax.set_xlabel('image ordinal (day-ordered)'); ax.set_ylabel(f'Z{TH_ZJ} [µm]')
ax.set_title(f'{TH_CORNER} Z{TH_ZJ} {nm} — time history ({WFS_NAME})')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

<a id='summary'></a>
## Per-Zj summary (r and slope vs Zj, all corners)

In [ ]:
alljs = sorted(df.j.unique())
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
for corner in CORNERS:
    rr, ss = [], []
    for j in alljs:
        sub = df[(df.corner == corner) & (df.j == j)]
        fit = robust_line(sub.fam_interp.to_numpy(), sub.cwfs_median.to_numpy())
        rr.append(fit['r']); ss.append(fit['slope'])
    a1.plot(alljs, rr, 'o-', ms=4, label=corner)
    a2.plot(alljs, ss, 'o-', ms=4, label=corner)
a1.set_xlabel('Noll j'); a1.set_ylabel('Pearson r'); a1.set_title('FAM–CWFS correlation')
a1.axhline(0, color='0.7', lw=0.6); a1.grid(alpha=0.3); a1.legend(fontsize=8)
a2.set_xlabel('Noll j'); a2.set_ylabel('OLS slope'); a2.set_title('CWFS-vs-FAM slope')
a2.axhline(1, color='0.7', lw=0.6); a2.grid(alpha=0.3); a2.legend(fontsize=8)
fig.suptitle(f'Per-Zernike FAM–CWFS summary — {WFS_NAME}', fontsize=11)
plt.show()